In [8]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/bananalatraichuoi
/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters
/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters/Powdery Mildew
/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters/Leaf Spot Disease
/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters/Anthracnose
/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters/Healthy
/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters/Leaf Blight
/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters/Sooty Mold


In [2]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader
import numpy as np
import logging
from PIL import Image
import warnings
import os
import joblib
import matplotlib.pyplot as plt
from itertools import cycle
from torch.utils.data import DataLoader, Subset

# Sklearn metrics
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             matthews_corrcoef, roc_auc_score, classification_report, confusion_matrix,
                             hamming_loss, cohen_kappa_score, jaccard_score, balanced_accuracy_score,
                             roc_curve, auc)

# Cấu hình hệ thống Kaggle
Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings('ignore', category=Image.DecompressionBombWarning)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# =====================
# CHỈNH SỬA ĐƯỜNG DẪN KAGGLE TẠI ĐÂY
# =====================
# Thay đổi dựa trên cấu trúc folder thực tế trong dataset của bạn
BASE_INPUT_PATH = "/kaggle/input/datasets/bananalatraichuoi/rambutan-uiters"

# Đường dẫn đầu ra (Bắt buộc dùng /kaggle/working/ để có quyền ghi)
MODEL_SAVE_PATH = "/kaggle/working/sklearn_models"
ROC_SAVE_PATH = "/kaggle/working/roc_curves"

# Tạo thư mục đầu ra
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(ROC_SAVE_PATH, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

def get_inception_extractor():
    model = models.inception_v3(weights='DEFAULT', aux_logits=True)
    model.fc = torch.nn.Identity()
    return model.to(DEVICE).eval()

def get_vgg19_extractor():
    model = models.vgg19(weights='DEFAULT')
    model.classifier = torch.nn.Sequential(*list(model.classifier.children())[:-1])
    return model.to(DEVICE).eval()

def extract_features(model, loader):
    features, labels = [], []
    with torch.no_grad():
        for imgs, y in loader:
            imgs = imgs.to(DEVICE)
            # Inception v3 yêu cầu input 299x299 và trả về tuple khi aux_logits=True
            out = model(imgs)
            if isinstance(out, torch.Tensor):
                features.append(out.cpu().numpy())
            else:
                features.append(out.logits.cpu().numpy())
            labels.append(y.numpy())
    return np.vstack(features), np.hstack(labels)

def plot_roc_curves(y_test, y_proba, class_names, classifier_name, feature_extractor_name):
    if y_proba is None: return
    n_classes = len(class_names)
    y_bin = label_binarize(y_test, classes=np.arange(n_classes))
    fpr, tpr, roc_auc = dict(), dict(), dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'brown'])
    for i, color in zip(range(n_classes), colors):
        ax.plot(fpr[i], tpr[i], color=color, lw=2, label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=2)
    ax.set_title(f'ROC Curves - {feature_extractor_name} + {classifier_name}')
    ax.legend(loc="lower right")
    plt.savefig(os.path.join(ROC_SAVE_PATH, f"roc_{feature_extractor_name}_{classifier_name}.png"))
    plt.close()

def run_reproduction_with_split():
    # Định nghĩa Transform (Theo bài báo: Inception v3 cần 299x299) [cite: 213, 271]
    transform_inc = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Load toàn bộ dữ liệu từ folder gốc
    full_dataset = datasets.ImageFolder(BASE_INPUT_PATH, transform=transform_inc)
    class_names = full_dataset.classes
    
    # 2. Thực hiện chia tập dữ liệu (80% Train, 20% Test) [cite: 9, 45]
    # Stratify giúp đảm bảo tỷ lệ các loại bệnh (chôm chôm, lúa...) cân bằng ở cả 2 tập [cite: 255]
    train_indices, test_indices = train_test_split(
        range(len(full_dataset)),
        test_size=0.2,
        random_state=42, # Cố định seed để kết quả tái lập được [cite: 546]
        stratify=full_dataset.targets
    )

    # Tạo các tập con (Subsets)
    train_subset = Subset(full_dataset, train_indices)
    test_subset = Subset(full_dataset, test_indices)

    # 3. Tạo DataLoader
    loader_train = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    loader_test = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    logger.info(f"📊 Tổng cộng: {len(full_dataset)} ảnh.")
    logger.info(f"✅ Đã chia: {len(train_subset)} ảnh để Train và {len(test_subset)} ảnh để Test.")

   # Bước 1: Trích xuất đặc trưng từ Inception V3
    model_inc = get_inception_extractor()
    X_inc_train, y_train = extract_features(model_inc, loader_train)
    X_inc_test, y_test = extract_features(model_inc, loader_test)
    
    logger.info("Đang trích xuất đặc trưng từ VGG19...")
    
    # Giải phóng GPU trước khi nạp model tiếp theo
    del model_inc
    torch.cuda.empty_cache()
    
    # Bước 1b: Trích xuất đặc trưng từ VGG19
    transform_vgg_final = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    model_vgg = get_vgg19_extractor()
    X_vgg_train, _ = extract_features(model_vgg, loader_train)
    X_vgg_test, _ = extract_features(model_vgg, loader_test)
    
    # Bước 2: Danh sách các bộ phân loại cần phục dựng
    classifiers = {
        "SVM": SVC(kernel='rbf', probability=True),
        "kNN": KNeighborsClassifier(n_neighbors=5),
        "RandomForest": RandomForestClassifier(n_estimators=100),
        "AdaBoost": AdaBoostClassifier(),
        "DecisionTree": DecisionTreeClassifier()
    }
    
    # Bước 3: Danh sách feature extractors
    feature_extractors = {
        "InceptionV3": (X_inc_train, X_inc_test),
        "VGG19": (X_vgg_train, X_vgg_test)
    }
    
    results = {}

    for extractor_name, (X_train, X_test) in feature_extractors.items():
        logger.info(f"\n{'='*60}\nĐang đánh giá {extractor_name}\n{'='*60}")
        results[extractor_name] = {}
        
        for name, clf in classifiers.items():
            logger.info(f"Đang đánh giá kết hợp: {extractor_name} + {name}")
            
            # Chuẩn hóa dữ liệu
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Train
            clf.fit(X_train_scaled, y_train)
            y_pred = clf.predict(X_test_scaled)
            y_proba = clf.predict_proba(X_test_scaled) if hasattr(clf, "predict_proba") else None
            
            # Tính toán metrics
            acc = accuracy_score(y_test, y_pred)
            pre = precision_score(y_test, y_pred, average='weighted')
            rec = recall_score(y_test, y_pred, average='weighted')
            f1 = f1_score(y_test, y_pred, average='weighted')
            mcc = matthews_corrcoef(y_test, y_pred)
            hamming = hamming_loss(y_test, y_pred)
            kappa = cohen_kappa_score(y_test, y_pred)
            jaccard = jaccard_score(y_test, y_pred, average='weighted')
            balanced_acc = balanced_accuracy_score(y_test, y_pred)
            
            auc_score = None
            if y_proba is not None:
                auc_score = roc_auc_score(y_test, y_proba, multi_class='ovr')
            
            # Lưu kết quả
            results[extractor_name][name] = {
                'acc': acc, 'pre': pre, 'rec': rec, 'f1': f1, 'mcc': mcc,
                'hamming': hamming, 'kappa': kappa, 'jaccard': jaccard,
                'balanced_acc': balanced_acc, 'auc': auc_score,
                'y_pred': y_pred, 'y_proba': y_proba
            }
            
            # In ra console
            print(f"\n{'='*60}")
            print(f"KẾT QUẢ: {extractor_name} + {name}")
            print(f"{'='*60}")
            print(f"Accuracy:           {acc:.4f}")
            print(f"Precision:          {pre:.4f}")
            print(f"Recall:             {rec:.4f}")
            print(f"F1 Score:           {f1:.4f}")
            print(f"MCC:                {mcc:.4f}")
            print(f"Cohen Kappa Score:  {kappa:.4f}")
            print(f"Balanced Accuracy:  {balanced_acc:.4f}")
            print(f"Jaccard Score:      {jaccard:.4f}")
            print(f"Hamming Loss:       {hamming:.4f}")
            if auc_score is not None:
                print(f"AUC Score:          {auc_score:.4f}")
            
            # Classification Report với per-class metrics
            print(f"\n{'='*60}")
            print("CLASSIFICATION REPORT (PER-CLASS METRICS)")
            print(f"{'='*60}")
            print(classification_report(y_test, y_pred, 
                                       target_names=class_names,
                                       digits=4))
            
            # Confusion Matrix
            print(f"\n{'='*60}")
            print("CONFUSION MATRIX")
            print(f"{'='*60}")
            cm = confusion_matrix(y_test, y_pred)
            print(cm)
            
            # ROC Curves
            if y_proba is not None:
                plot_roc_curves(y_test, y_proba, class_names, name, extractor_name)
            
            # Lưu model tốt nhất
            model_filename = os.path.join(MODEL_SAVE_PATH, f"{extractor_name}_{name}_model.joblib")
            scaler_filename = os.path.join(MODEL_SAVE_PATH, f"{extractor_name}_{name}_scaler.joblib")
            joblib.dump(clf, model_filename)
            joblib.dump(scaler, scaler_filename)
            logger.info(f"✓ Lưu model: {model_filename}")
            logger.info(f"✓ Lưu scaler: {scaler_filename}")

    logger.info("Hoàn tất phục dựng mô hình.")
    
    # === LƯU BÁNG CÓ KỂTQUẢ ===
    summary_file = os.path.join(MODEL_SAVE_PATH, "evaluation_summary.txt")
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("BÁNG CÓ KỂTQUẢ ĐÁNH GIÁ MÔ HÌNH - INCEPTION V3 + VGG19 + SKLEARN\n")
        f.write("="*80 + "\n\n")
        
        for extractor_name, classifiers_results in results.items():
            f.write(f"\n{'='*80}\n")
            f.write(f"FEATURE EXTRACTOR: {extractor_name}\n")
            f.write(f"{'='*80}\n")
            
            for classifier_name, metrics_dict in classifiers_results.items():
                f.write(f"\n{'-'*80}\n")
                f.write(f"CLASSIFIER: {classifier_name}\n")
                f.write(f"{'-'*80}\n")
                f.write(f"Accuracy:           {metrics_dict['acc']:.4f}\n")
                f.write(f"Precision:          {metrics_dict['pre']:.4f}\n")
                f.write(f"Recall:             {metrics_dict['rec']:.4f}\n")
                f.write(f"F1 Score:           {metrics_dict['f1']:.4f}\n")
                f.write(f"Matthews Corr Coef: {metrics_dict['mcc']:.4f}\n")
                f.write(f"Cohen Kappa Score:  {metrics_dict['kappa']:.4f}\n")
                f.write(f"Balanced Accuracy:  {metrics_dict['balanced_acc']:.4f}\n")
                f.write(f"Jaccard Score:      {metrics_dict['jaccard']:.4f}\n")
                f.write(f"Hamming Loss:       {metrics_dict['hamming']:.4f}\n")
                if metrics_dict['auc'] is not None and not np.isnan(metrics_dict['auc']):
                    f.write(f"AUC Score:          {metrics_dict['auc']:.4f}\n")
    
    logger.info(f"✓ Báng cóc kểtquả được lưu: {summary_file}")

if __name__ == "__main__":
    run_reproduction_with_split()

2026-04-04 03:31:42,237 - 📊 Tổng cộng: 1076 ảnh.
2026-04-04 03:31:42,238 - ✅ Đã chia: 860 ảnh để Train và 216 ảnh để Test.


Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 204MB/s]  
2026-04-04 03:36:45,316 - Đang trích xuất đặc trưng từ VGG19...


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 202MB/s] 
2026-04-04 03:41:35,086 - 
Đang đánh giá InceptionV3
2026-04-04 03:41:35,087 - Đang đánh giá kết hợp: InceptionV3 + SVM


...


KẾT QUẢ: VGG19 + kNN
Accuracy:           0.2083
Precision:          0.1867
Recall:             0.2083
F1 Score:           0.1958
MCC:                0.0124
Cohen Kappa Score:  0.0124
Balanced Accuracy:  0.1726
Jaccard Score:      0.1143
Hamming Loss:       0.7917
AUC Score:          0.4918

CLASSIFICATION REPORT (PER-CLASS METRICS)
                   precision    recall  f1-score   support

      Anthracnose     0.0000    0.0000    0.0000        10
          Healthy     0.2571    0.2903    0.2727        31
      Leaf Blight     0.3390    0.4444    0.3846        45
Leaf Spot Disease     0.0000    0.0000    0.0000        30
   Powdery Mildew     0.1143    0.0976    0.1053        41
       Sooty Mold     0.2105    0.2034    0.2069        59

         accuracy                         0.2083       216
        macro avg     0.1535    0.1726    0.1616       216
     weighted avg     0.1867    0.2083    0.1958       216


CONFUSION MATRIX
[[ 0  2  5  0  1  2]
 [ 2  9  1  7  6  6]
 [ 4  1 20  

2026-04-04 03:42:20,103 - ✓ Lưu model: /kaggle/working/sklearn_models/VGG19_RandomForest_model.joblib
2026-04-04 03:42:20,104 - ✓ Lưu scaler: /kaggle/working/sklearn_models/VGG19_RandomForest_scaler.joblib
2026-04-04 03:42:20,104 - Đang đánh giá kết hợp: VGG19 + AdaBoost
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetr


KẾT QUẢ: VGG19 + AdaBoost
Accuracy:           0.2639
Precision:          0.2145
Recall:             0.2639
F1 Score:           0.2274
MCC:                0.0445
Cohen Kappa Score:  0.0420
Balanced Accuracy:  0.1959
Jaccard Score:      0.1338
Hamming Loss:       0.7361
AUC Score:          0.5362

CLASSIFICATION REPORT (PER-CLASS METRICS)
                   precision    recall  f1-score   support

      Anthracnose     0.0000    0.0000    0.0000        10
          Healthy     0.2000    0.0968    0.1304        31
      Leaf Blight     0.3333    0.3111    0.3218        45
Leaf Spot Disease     0.0000    0.0000    0.0000        30
   Powdery Mildew     0.2400    0.2927    0.2637        41
       Sooty Mold     0.2593    0.4746    0.3353        59

         accuracy                         0.2639       216
        macro avg     0.1721    0.1959    0.1752       216
     weighted avg     0.2145    0.2639    0.2274       216


CONFUSION MATRIX
[[ 0  2  2  0  2  4]
 [ 0  3  1  1  7 19]
 [ 0  1

2026-04-04 03:42:30,060 - ✓ Lưu model: /kaggle/working/sklearn_models/VGG19_AdaBoost_model.joblib
2026-04-04 03:42:30,061 - ✓ Lưu scaler: /kaggle/working/sklearn_models/VGG19_AdaBoost_scaler.joblib
2026-04-04 03:42:30,061 - Đang đánh giá kết hợp: VGG19 + DecisionTree
2026-04-04 03:42:32,332 - ✓ Lưu model: /kaggle/working/sklearn_models/VGG19_DecisionTree_model.joblib
2026-04-04 03:42:32,333 - ✓ Lưu scaler: /kaggle/working/sklearn_models/VGG19_DecisionTree_scaler.joblib
2026-04-04 03:42:32,334 - Hoàn tất phục dựng mô hình.
2026-04-04 03:42:32,336 - ✓ Báng cóc kểtquả được lưu: /kaggle/working/sklearn_models/evaluation_summary.txt



KẾT QUẢ: VGG19 + DecisionTree
Accuracy:           0.1713
Precision:          0.1699
Recall:             0.1713
F1 Score:           0.1703
MCC:                -0.0343
Cohen Kappa Score:  -0.0343
Balanced Accuracy:  0.1367
Jaccard Score:      0.0947
Hamming Loss:       0.8287
AUC Score:          0.4796

CLASSIFICATION REPORT (PER-CLASS METRICS)
                   precision    recall  f1-score   support

      Anthracnose     0.0000    0.0000    0.0000        10
          Healthy     0.1200    0.0968    0.1071        31
      Leaf Blight     0.2000    0.2000    0.2000        45
Leaf Spot Disease     0.0541    0.0667    0.0597        30
   Powdery Mildew     0.2093    0.2195    0.2143        41
       Sooty Mold     0.2333    0.2373    0.2353        59

         accuracy                         0.1713       216
        macro avg     0.1361    0.1367    0.1361       216
     weighted avg     0.1699    0.1713    0.1703       216


CONFUSION MATRIX
[[ 0  2  2  1  3  2]
 [ 0  3  1  8 11  8]
 